# Magnetostatics

The magnetic field of a wire coil, computed in two steps:

* first, we solve an electric conduction problem in the wire
* then, the computed current density is the source for the magnetostatic Maxwell equations

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

## Model of the coil

A helix on a cylinder surface, connected to straight leads by Bézier curves, and swept with a circular cross-section:

In [ ]:
cyl = Cylinder((0,0,0), Z, r=0.01, h=0.03).faces[0]
heli = Edge(Segment((0,0), (12*pi, 0.03)), cyl)
ps = heli.start
vs = heli.start_tangent
pe = heli.end
ve = heli.end_tangent

e1 = Segment((0,0,-0.03), (0,0,-0.01))
c1 = BezierCurve( [(0,0,-0.01), (0,0,0), ps-vs, ps])
e2 = Segment((0,0,0.04), (0,0,0.06))
c2 = BezierCurve( [pe, pe+ve, (0,0,0.03), (0,0,0.04)])
spiral = Wire([e1, c1, heli, c2, e2])
circ = Face(Wire([Circle((0,0,-0.03), Z, 0.001)]))
coil = Pipe(spiral, circ)

coil.faces.maxh=0.2
coil.faces.name="coilbnd"
coil.faces.Max(Z).name="in"
coil.faces.Min(Z).name="out"
coil.faces.col=(184/256, 115/256, 51/256)
coil.mat("coil")
crosssection = coil.faces.Max(Z).mass

In [ ]:
ea = { "euler_angles" : (-130, -73, 0), "radius" : 0.025 }
Draw (coil, **ea);

an air box around the coil:

In [ ]:
box = Box((-0.04,-0.04,-0.03), (0.04,0.04,0.06))
box.faces.name = "outer"
air = box-coil
air.mat("air");

## Mesh generation

curved tetrahedral mesh of coil and air box:

In [ ]:
with TaskManager():
    mesh = Glue([coil,air]).GenerateMesh(meshsize.coarse, maxh=0.01).Curve(3)

In [ ]:
clipping = { "clipping" : { "y":1, "z":0, "dist":0.012} }
Draw (mesh, **clipping, **ea);

In [ ]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

## Current density in the wire
on the domain $\Omega_{\text{coil}}$, solve for potential $\Phi$ and current density $j$:

\begin{align*}
j & = \sigma \nabla \Phi \\
\operatorname{div} j & = 0
\end{align*}
with electric conductivity $\sigma$.


port boundary conditions: 
\begin{align*}
\Phi & = 0  \qquad \qquad \text{on } \Gamma_{\text{out}},  \\
j_n & = \frac{1}{|\Gamma_{in}|} \quad \qquad \text{on } \Gamma_{\text{in}},
\end{align*}

and $j_n=0$ else

In [ ]:
sigma = 58.7e6
phi,psi = H1(mesh, order=3, definedon="coil").TnT()
with TaskManager():
    gfphi = Solve(sigma*grad(phi)*grad(psi)*dx == 1/crosssection*psi*ds("in"), phi[BND("out")]==0)

In [ ]:
Draw (gfphi, draw_vol=False, **clipping, **ea);

## Magnetostatics

The magnetic field $H$ and the magnetic flux density $B$ satisfy

$$
\operatorname{curl} H = j, \qquad \operatorname{div} B = 0, \qquad B = \mu H.
$$

Since $B$ is divergence-free, it is the curl of a vector potential, $B = \operatorname{curl} u$, which leads to $\operatorname{curl} \mu^{-1} \operatorname{curl} u = j$.
The vector potential $u \in H(\operatorname{curl})$ is discretized by high-order Nédélec elements: find $u$ such that

$$
\int \mu^{-1} \operatorname{curl} u \cdot \operatorname{curl} v \, dx =
\int j \cdot v \, dx \qquad \forall \, v.
$$

A small regularization term makes the problem uniquely solvable; CG with BDDC preconditioning solves it:

In [ ]:
mu = 4*pi*1e-7
u,v = HCurl(mesh, order=2, nograds=True).TnT()
with TaskManager():
    gfu = Solve(1/mu*curl(u)*curl(v)*dx + 1e-6/mu*u*v*dx == sigma*grad(gfphi)*v*dx("coil"),
                preconditioners.BDDC(), solvers.CGSolver.Creator(plotrates=True))

Check: the magnetic flux density $B = \operatorname{curl} u$ in the center, compared to the formula for a finite solenoid with $N$ turns, length $L$ and radius $r$, $B = \mu N I / \sqrt{L^2 + 4 r^2}$:

In [ ]:
N, L, r = 6, 0.03, 0.01
print ("FEM:    ", curl(gfu)(mesh(0,0,0.015))[2])
print ("formula:", mu*N/sqrt(L**2+4*r**2))

## Field lines

In [ ]:
s = 0.04*2
N = 15 
p = [(-s+2*s*i/N,-s+2*s*j/N,-s+2*s*k/N) for i in range(1,N) for j in range(1,N) for k in range(1,N)]
fieldlines = curl(gfu)._BuildFieldLines(mesh, p, num_fieldlines=N**3//5, randomized=True, length=0.3)

Draw(curl(gfu), mesh,  "X", draw_vol=False, draw_surf=True, objects=[fieldlines], \
     min=0, max=1e-4, autoscale=False, settings={"Objects": {"Surface": False}},
     **ea, **clipping);